[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/23.%20The%20AGV%20Dispatching%20%26%20Battery%20Management%20Problem/P23-Tier-3-BACKUP_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# Problem 23: The AGV Dispatching and Battery Management Problem

## Tier 3: Ant Colony Optimization for AGV Dispatching

### Goal and workflow

This notebook uses Ant Colony Optimization (ACO), a population-based metaheuristic, to construct and improve dispatch plans for automated guided vehicles (AGVs). Each ant builds a plan by assigning transport tasks to AGVs and selecting the order in which each AGV visits pickup, drop-off, and charging locations. Pheromone trails and local heuristic information influence those choices.

The notebook's workflow is: define the terminal instance; calculate travel-time and energy data; construct and evaluate ant solutions; update pheromones over generations; inspect the best feasible solution and search history; and examine the resulting routes and pheromone distribution.

### Scope and modeling assumptions

- The terminal is represented by locations in a two-dimensional Euclidean plane. Travel time is proportional to Euclidean distance, and energy use is proportional to distance.
- The travel-time matrix represents travel at unit speed. Each AGV's travel time is obtained by dividing this value by its speed; energy use is independent of speed in this model.
- Every task is assigned to exactly one AGV. Its pickup must occur before its drop-off, and each task node is visited once.
- An AGV starts at the depot with its specified initial battery and returns to the depot after its assigned tasks. At a charging station, it charges instantaneously to full capacity.
- Battery feasibility means that the battery must not fall below the AGV's minimum reserve during any travel leg, including the final return to the depot. The construction rule biases choices toward charging stations when a proposed leg would cross that reserve; this bias is not a guarantee of feasibility.
- Time windows, deadlines, service durations, charging durations, station capacity, traffic, and interactions between AGVs are not enforced. Although some data structures contain task time-window and service fields and charging rates, those fields do not enter the objective or feasibility tests in this example.
- The objective is the sum of AGV travel times. Charging itself has no modeled time or cost. A feasible plan is one with no battery-reserve violations.

These assumptions define a compact dispatching example rather than a complete operational model. ACO is a stochastic search method: it can find good feasible plans, but it does not certify global optimality.

### ACO concepts and mathematical model

An ant is a simulated search agent that constructs one complete dispatch plan. Pheromone, written as $\tau_{ij}$, is a numerical memory associated with moving from location $i$ to location $j$. The heuristic value $\eta_{ij}$ reflects the local attractiveness of that move; here, shorter travel time and lower energy use produce a larger value. The parameters $\alpha$ and $\beta$ control the relative influence of pheromone and heuristic information. This probabilistic construction allows the colony to explore different plans while learning from routes used by feasible plans.

Let $V$ be the set of terminal nodes, $K$ the set of AGVs, and $A$ the set of ants. For a candidate move from the current node $i$ to an eligible unvisited node $j$, the construction weight is

$$w_{ij} = \tau_{ij}^{\alpha}\eta_{ij}^{\beta}b_{ij},$$

and the selection probability is

$$p_{ij} = \frac{w_{ij}}{\sum_{\ell \in C} w_{i\ell}},$$

where $C$ is the set of currently eligible next nodes and $b_{ij}$ is a battery-aware factor. The code uses a smaller factor when the move is predicted to cross the battery reserve, and a larger factor for a charging station in that situation. These are soft preferences: the final route evaluation, rather than the probability rule, determines battery feasibility. The code implements these quantities in `calculate_transition_probability` and the roulette-wheel selection in `build_agv_route`.

For a feasible solution $s$ with total travel-time cost $L_s$, pheromone evaporates and feasible routes deposit pheromone according to

$$\tau_{ij} \leftarrow (1-\rho)\tau_{ij} + \sum_{s \in S_{\mathrm{feasible}}} \frac{q}{L_s}\,\mathbf{1}\{(i,j) \text{ is used by }s\},$$

where $\rho$ is the evaporation rate, $q$ is the deposition constant, and $\mathbf{1}\{\cdot\}$ is one when the condition is true and zero otherwise. Evaporation limits the influence of old search history; deposition reinforces edges used by feasible plans. The implementation is `update_pheromones`.

The plan objective is $L=\sum_{k\in K} L_k$, where $L_k$ is the travel time along AGV $k$'s route, including its return to the depot. The feasibility condition is that the battery reserve is respected after every travel leg. The implementation evaluates these quantities in `evaluate_route`. The search retains the lowest-cost feasible plan encountered, not necessarily the globally optimal plan.

If $G$ is the number of generations, $A$ the number of ants, and $|V|$ the number of nodes, route construction checks candidate moves repeatedly and takes up to quadratic work in $|V|$ per ant in this implementation. Pairwise solution-diversity calculation adds work proportional to $A^2|V|$ per generation. The pheromone table stores up to $|V|(|V|-1)$ directed edges. These are broad upper-bound descriptions; actual work depends on the assigned tasks and available candidates.

### Example instance

The instance below contains six transport tasks, three AGVs, two charging stations, and twenty ants searched over one hundred generations. The parameter values are instructional inputs, not evidence of a particular solution quality. Results should be interpreted from the execution and plots produced by running the notebook.

In [ ]:
# Import the libraries used for the data model, search, and visualizations.
import random
import time
from dataclasses import dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Fixed seeds make the stochastic example reproducible in a fresh run.
np.random.seed(42)
random.seed(42)

plt.style.use("default")
sns.set_palette("husl")

In [ ]:
@dataclass
class Task:
    """Transport task data; only pickup, drop-off, and ID affect this model."""

    id: str
    pickup: str
    dropoff: str
    time_window: Tuple[float, float]
    service_time: float
    priority: float = 1.0
    deadline: float = float("inf")


@dataclass
class AGV:
    """Vehicle capacity, reserve, initial charge, and travel speed."""

    id: str
    battery_capacity: float
    battery_min: float
    initial_battery: float
    speed: float = 1.0


@dataclass
class ChargingStation:
    """Charging-station data; charging rate is not used in this model."""

    id: str
    charging_rate: float


@dataclass
class Location:
    """A terminal location represented by its planar coordinates."""

    id: str
    x: float
    y: float

    def distance_to(self, other: "Location") -> float:
        """Return Euclidean distance in the coordinate system's units."""
        return float(np.hypot(self.x - other.x, self.y - other.y))


@dataclass
class AntSolution:
    """Routes and evaluation measures for one constructed ant solution."""

    routes: Dict[str, List[str]]
    cost: float
    feasible: bool
    battery_violations: int
    time_violations: int


@dataclass
class PheromoneInfo:
    """Pheromone-trail parameters and directed-edge trail values."""

    trails: Dict[Tuple[str, str], float]
    evaporation_rate: float = 0.1
    deposition_constant: float = 1.0

In [ ]:
# Define terminal locations in a two-dimensional Euclidean plane.
locations = {
    "O": Location("O", 0, 0),
    "P1": Location("P1", 2, 3),
    "D1": Location("D1", 6, 4),
    "P2": Location("P2", 1, 5),
    "D2": Location("D2", 5, 7),
    "P3": Location("P3", 3, 1),
    "D3": Location("D3", 7, 2),
    "P4": Location("P4", 4, 6),
    "D4": Location("D4", 8, 5),
    "P5": Location("P5", 2, 7),
    "D5": Location("D5", 6, 8),
    "P6": Location("P6", 5, 3),
    "D6": Location("D6", 9, 4),
    "C1": Location("C1", 3, 3),
    "C2": Location("C2", 6, 1),
}

# Time windows, service times, priorities, and deadlines are retained as
# task data, although this example's objective and feasibility tests do not use them.
tasks = [
    Task("T1", "P1", "D1", (0, 30), 2.0, priority=1.0, deadline=35),
    Task("T2", "P2", "D2", (5, 25), 1.5, priority=0.8, deadline=30),
    Task("T3", "P3", "D3", (10, 40), 2.5, priority=1.2, deadline=45),
    Task("T4", "P4", "D4", (15, 35), 1.8, priority=0.9, deadline=40),
    Task("T5", "P5", "D5", (8, 32), 2.2, priority=1.1, deadline=38),
    Task("T6", "P6", "D6", (12, 42), 1.6, priority=0.7, deadline=48),
]

agvs = [
    AGV("K1", 100.0, 20.0, 100.0, speed=1.0),
    AGV("K2", 120.0, 25.0, 120.0, speed=1.2),
    AGV("K3", 90.0, 15.0, 90.0, speed=0.9),
]

charging_stations = [
    ChargingStation("C1", 8.0),
    ChargingStation("C2", 12.0),
]


def calculate_matrices(locations):
    """Build unit-speed travel-time and distance-based energy matrices."""
    travel_times = {}
    energy_consumption = {}

    for loc1_id, loc1 in locations.items():
        for loc2_id, loc2 in locations.items():
            distance = loc1.distance_to(loc2)
            travel_times[(loc1_id, loc2_id)] = distance
            energy_consumption[(loc1_id, loc2_id)] = distance * 1.2

    return travel_times, energy_consumption


travel_times, energy_consumption = calculate_matrices(locations)

print("AGV Dispatching Instance")
print(f"Tasks: {len(tasks)}")
print(f"AGVs: {len(agvs)}")
print(f"Charging stations: {len(charging_stations)}")
print(f"Locations: {list(locations)}")

print("\nTask data:")
for task in tasks:
    print(
        f"  {task.id}: priority={task.priority:.1f}, "
        f"deadline={task.deadline}, time window={task.time_window}"
    )

## Constructing and evaluating ant solutions

Each ant first assigns every task to one AGV at random. It then builds one route per AGV. To preserve task logic, a task's pickup is eligible until visited, and its drop-off becomes eligible only after that pickup. Charging stations may also be considered, but each station can appear at most once in a given route. The route begins and ends at the depot.

At each step, roulette-wheel selection draws a candidate in proportion to its normalized weight. A high weight makes a move more likely, not certain. During evaluation, each travel leg consumes energy; arriving at a charging station restores the battery to capacity after checking whether the arrival itself violated the reserve. The route cost is travel time adjusted for the AGV's speed. Feasibility is determined by the number of battery-reserve violations.

In [ ]:
class AGVAntColonyOptimizer:
    """ACO search for task dispatching with battery-reserve feasibility."""

    def __init__(
        self,
        locations,
        travel_times,
        energy_consumption,
        charging_stations,
        agvs,
        tasks,
        num_ants=20,
        generations=100,
        alpha=1.0,
        beta=2.0,
        rho=0.1,
        q=1.0,
    ):
        self.locations = locations
        self.travel_times = travel_times
        self.energy_consumption = energy_consumption
        self.charging_stations = charging_stations
        self.agvs = agvs
        self.tasks = tasks

        self.num_ants = num_ants
        self.generations = generations
        self.alpha = alpha
        self.beta = beta
        self.rho = rho
        self.q = q

        self.nodes = ["O"]
        for task in tasks:
            self.nodes.extend([task.pickup, task.dropoff])
        self.nodes.extend(station.id for station in charging_stations)

        self.pheromone = self.initialize_pheromones()
        self.heuristic = self.calculate_heuristic_info()
        self.best_solution = None
        self.best_cost = float("inf")
        self.convergence_history = []
        self.diversity_history = []

    def initialize_pheromones(self):
        """Give each directed edge the same positive initial trail."""
        tau_0 = 1.0 / len(self.nodes)
        return {
            (node_i, node_j): tau_0
            for node_i in self.nodes
            for node_j in self.nodes
            if node_i != node_j
        }

    def calculate_heuristic_info(self):
        """Use inverse unit-speed travel time plus weighted energy as attractiveness."""
        heuristic = {}

        for node_i in self.nodes:
            for node_j in self.nodes:
                if node_i == node_j:
                    continue

                travel_time = self.travel_times[(node_i, node_j)]
                energy = self.energy_consumption[(node_i, node_j)]
                cost = travel_time + 0.1 * energy
                heuristic[(node_i, node_j)] = 1.0 / cost if cost > 0 else 0.0

        return heuristic

    def calculate_transition_probability(
        self, current_node, next_node, visited_nodes, agv, current_battery
    ):
        """Return an unnormalized ACO weight for one eligible move."""
        if next_node in visited_nodes:
            return 0.0

        tau = self.pheromone[(current_node, next_node)]
        eta = self.heuristic[(current_node, next_node)]
        energy_needed = self.energy_consumption[(current_node, next_node)]
        battery_factor = 1.0

        if current_battery - energy_needed < agv.battery_min:
            battery_factor = 0.1
            if next_node in {station.id for station in self.charging_stations}:
                battery_factor = 5.0

        return (tau ** self.alpha) * (eta ** self.beta) * battery_factor

    def construct_ant_solution(self, ant_id):
        """Assign tasks and construct one precedence-respecting plan."""
        routes = {agv.id: ["O"] for agv in self.agvs}
        task_assignments = {
            task.id: random.choice(self.agvs).id for task in self.tasks
        }
        total_cost = 0.0
        total_violations = 0

        for agv in self.agvs:
            assigned_tasks = [
                task for task in self.tasks if task_assignments[task.id] == agv.id
            ]
            route = self.build_agv_route(agv, assigned_tasks)
            routes[agv.id] = route
            route_cost, violations = self.evaluate_route(route, agv)
            total_cost += route_cost
            total_violations += violations

        return AntSolution(
            routes=routes,
            cost=total_cost,
            feasible=(total_violations == 0),
            battery_violations=total_violations,
            time_violations=0,
        )

    def build_agv_route(self, agv, tasks):
        """Construct one AGV route with pickup-before-drop-off eligibility."""
        route = ["O"]
        current_node = "O"
        current_battery = agv.initial_battery
        visited_nodes = {"O"}
        picked_up = set()
        completed = set()

        while len(completed) < len(tasks):
            candidates = []
            for task in tasks:
                if task.id not in picked_up and task.pickup not in visited_nodes:
                    candidates.append(task.pickup)
                elif task.id in picked_up and task.id not in completed:
                    candidates.append(task.dropoff)

            candidates.extend(
                station.id
                for station in self.charging_stations
                if station.id not in visited_nodes
            )

            weights = {
                node: self.calculate_transition_probability(
                    current_node, node, visited_nodes, agv, current_battery
                )
                for node in candidates
            }
            total_weight = sum(weights.values())

            if total_weight <= 0.0:
                next_node = random.choice(candidates)
            else:
                threshold = random.random() * total_weight
                cumulative_weight = 0.0
                next_node = candidates[-1]
                for candidate in candidates:
                    cumulative_weight += weights[candidate]
                    if threshold <= cumulative_weight:
                        next_node = candidate
                        break

            route.append(next_node)
            visited_nodes.add(next_node)
            current_battery -= self.energy_consumption[(current_node, next_node)]

            for task in tasks:
                if next_node == task.pickup:
                    picked_up.add(task.id)
                elif next_node == task.dropoff:
                    completed.add(task.id)

            if next_node in {station.id for station in self.charging_stations}:
                current_battery = agv.battery_capacity

            current_node = next_node

        route.append("O")
        return route

    def evaluate_route(self, route, agv):
        """Return travel-time cost and battery-reserve violations for a route."""
        total_cost = 0.0
        current_battery = agv.initial_battery
        violations = 0
        station_ids = {station.id for station in self.charging_stations}

        for from_node, to_node in zip(route, route[1:]):
            total_cost += self.travel_times[(from_node, to_node)] / agv.speed
            current_battery -= self.energy_consumption[(from_node, to_node)]

            # Check arrival charge before recharging so a low-charge arrival
            # cannot be hidden by the station's full-charge reset.
            if current_battery < agv.battery_min:
                violations += 1

            if to_node in station_ids:
                current_battery = agv.battery_capacity

        return total_cost, violations

    def update_pheromones(self, ant_solutions):
        """Evaporate trails and reinforce edges in feasible ant solutions."""
        for edge in self.pheromone:
            self.pheromone[edge] *= 1.0 - self.rho

        for solution in ant_solutions:
            if not solution.feasible or solution.cost <= 0.0:
                continue

            deposit_amount = self.q / solution.cost
            for route in solution.routes.values():
                for edge in zip(route, route[1:]):
                    if edge in self.pheromone:
                        self.pheromone[edge] += deposit_amount

    def calculate_solution_diversity(self, ant_solutions):
        """Return mean pairwise route-set distance for the generation."""
        if len(ant_solutions) < 2:
            return 0.0

        distances = []
        for index, solution in enumerate(ant_solutions):
            for other_solution in ant_solutions[index + 1:]:
                distances.append(
                    self.calculate_solution_distance(solution, other_solution)
                )

        return float(np.mean(distances))

    def calculate_solution_distance(self, sol1, sol2):
        """Compare the sets of visited nodes assigned to each AGV.

        This diversity measure ignores visit order and repeated depot visits.
        """
        distance = 0.0
        for agv in self.agvs:
            set1 = set(sol1.routes.get(agv.id, [])) - {"O"}
            set2 = set(sol2.routes.get(agv.id, [])) - {"O"}
            denominator = max(len(set1), len(set2))
            if denominator:
                distance += len(set1.symmetric_difference(set2)) / denominator

        return distance / len(self.agvs) if self.agvs else 0.0

    def optimize(self):
        """Run the specified ant-construction and pheromone-update cycle."""
        print(
            f"Starting ACO with {self.num_ants} ants "
            f"for {self.generations} generations..."
        )

        for generation in range(self.generations):
            ant_solutions = [
                self.construct_ant_solution(ant_id)
                for ant_id in range(self.num_ants)
            ]
            feasible_solutions = [
                solution for solution in ant_solutions if solution.feasible
            ]

            if feasible_solutions:
                generation_best = min(feasible_solutions, key=lambda item: item.cost)
                if generation_best.cost < self.best_cost:
                    self.best_solution = generation_best
                    self.best_cost = generation_best.cost
                average_cost = float(
                    np.mean([solution.cost for solution in feasible_solutions])
                )
            else:
                average_cost = float("nan")

            self.update_pheromones(ant_solutions)
            diversity = self.calculate_solution_diversity(ant_solutions)
            self.convergence_history.append(
                {
                    "generation": generation + 1,
                    "best_cost": self.best_cost,
                    "avg_cost": average_cost,
                    "feasible_count": len(feasible_solutions),
                }
            )
            self.diversity_history.append(diversity)

            if (generation + 1) % 20 == 0:
                average_text = (
                    f"{average_cost:.2f}"
                    if np.isfinite(average_cost)
                    else "not available"
                )
                best_text = (
                    f"{self.best_cost:.2f}"
                    if self.best_solution is not None
                    else "not available"
                )
                print(
                    f"  Generation {generation + 1:3d}: "
                    f"Best={best_text}, Average={average_text}, "
                    f"Feasible={len(feasible_solutions)}/{self.num_ants}"
                )

        if self.best_solution is None:
            print("ACO completed without finding a feasible solution.")
        else:
            print(f"ACO completed. Best feasible cost: {self.best_cost:.2f}")

        return (
            self.best_solution,
            self.convergence_history,
            self.diversity_history,
        )

## Running the search

The optimizer uses the instance and parameter values defined above. A reported best cost is the lowest feasible travel-time cost encountered during the search. If no feasible ant solution is found, the result is `None`; this is a search outcome, not proof that the model has no feasible plan.

In [ ]:
aco = AGVAntColonyOptimizer(
    locations=locations,
    travel_times=travel_times,
    energy_consumption=energy_consumption,
    charging_stations=charging_stations,
    agvs=agvs,
    tasks=tasks,
    num_ants=20,
    generations=100,
    alpha=1.0,
    beta=2.0,
    rho=0.1,
    q=1.0,
)

start_time = time.time()
best_solution, convergence_history, diversity_history = aco.optimize()
computation_time = time.time() - start_time
print(f"\nSearch time: {computation_time:.2f} seconds")

print("\n" + "=" * 60)
print("BEST FEASIBLE ACO SOLUTION")
print("=" * 60)

if best_solution is None:
    print("No feasible solution was found during this run.")
else:
    print(f"Total travel-time cost: {best_solution.cost:.2f}")
    print(f"Feasible: {best_solution.feasible}")
    print(f"Battery-reserve violations: {best_solution.battery_violations}")

    agvs_by_id = {agv.id: agv for agv in agvs}
    for agv_id, route in best_solution.routes.items():
        if len(route) <= 2:
            continue
        route_cost, violations = aco.evaluate_route(route, agvs_by_id[agv_id])
        print(f"\nAGV {agv_id}")
        print(f"  Route: {' -> '.join(route)}")
        print(f"  Travel-time cost: {route_cost:.2f}")
        print(f"  Battery-reserve violations: {violations}")

    # Verify that the stored objective equals the sum of route evaluations.
    reconstructed_cost = sum(
        aco.evaluate_route(route, agvs_by_id[agv_id])[0]
        for agv_id, route in best_solution.routes.items()
    )
    assert np.isclose(reconstructed_cost, best_solution.cost)
    assert best_solution.battery_violations == sum(
        aco.evaluate_route(route, agvs_by_id[agv_id])[1]
        for agv_id, route in best_solution.routes.items()
    )

## Reference heuristic comparison

The following simple rule is included as a reference point, not as a second optimization method. It appends each task's pickup and drop-off to the route of the first AGV with the lowest standalone task travel time. The task's standalone cost is identical for every AGV in this model, so ties are resolved by AGV list order. This rule does not optimize task sequences jointly and does not insert charging stops; its battery violations are reported rather than repaired. The comparison is descriptive and does not establish that ACO will outperform the reference on every run or instance.

In [ ]:
def compare_with_heuristic(aco, best_solution, tasks, agvs, travel_times):
    """Compare ACO with a simple standalone-task assignment reference."""
    print("\n" + "=" * 60)
    print("COMPARISON WITH SIMPLE REFERENCE RULE")
    print("=" * 60)

    routes = {agv.id: ["O"] for agv in agvs}
    for task in tasks:
        best_agv = min(
            agvs,
            key=lambda agv: (
                travel_times[("O", task.pickup)]
                + travel_times[(task.pickup, task.dropoff)]
                + travel_times[(task.dropoff, "O")]
            )
            / agv.speed,
        )
        routes[best_agv.id].extend([task.pickup, task.dropoff])

    for route in routes.values():
        if len(route) > 1:
            route.append("O")

    agvs_by_id = {agv.id: agv for agv in agvs}
    reference_cost = 0.0
    reference_violations = 0
    for agv_id, route in routes.items():
        if len(route) > 2:
            route_cost, violations = aco.evaluate_route(route, agvs_by_id[agv_id])
            reference_cost += route_cost
            reference_violations += violations

    print("Simple reference rule:")
    print(f"  Total travel-time cost: {reference_cost:.2f}")
    print(f"  Battery-reserve violations: {reference_violations}")

    if best_solution is None:
        aco_cost = float("nan")
        aco_violations = None
        cost_improvement = float("nan")
        print("\nNo feasible ACO solution is available for comparison.")
    else:
        aco_cost = best_solution.cost
        aco_violations = best_solution.battery_violations
        cost_improvement = (
            (reference_cost - aco_cost) / reference_cost * 100.0
            if reference_cost > 0
            else float("nan")
        )
        print("\nBest feasible ACO solution:")
        print(f"  Total travel-time cost: {aco_cost:.2f}")
        print(f"  Battery-reserve violations: {aco_violations}")
        if np.isfinite(cost_improvement):
            print(f"  Cost difference relative to reference: {cost_improvement:.1f}%")

    return {
        "heuristic_cost": reference_cost,
        "aco_cost": aco_cost,
        "heuristic_violations": reference_violations,
        "aco_violations": aco_violations,
        "cost_improvement": cost_improvement,
    }


comparison_results = compare_with_heuristic(
    aco, best_solution, tasks, agvs, travel_times
)

## Visualizing route and search behavior

The route panel shows the locations and the best feasible routes, when one is available. The convergence panel reports the best feasible cost found so far, the average cost among feasible ants in each generation, and the number of feasible ants. If a generation contains no feasible ant, its average feasible cost is undefined and is left blank in the plot. The diversity statistic compares sets of visited nodes by AGV; it does not measure differences in visit order.

In [ ]:
def visualize_aco_results(
    locations,
    agvs,
    best_solution,
    convergence_history,
    diversity_history,
    comparison_results,
):
    """Plot routes, generation metrics, diversity, and reference costs."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("Ant Colony Optimization: Search and Route Analysis", fontsize=14)

    # Terminal map: marker shape identifies location type; color identifies AGV.
    ax1 = axes[0, 0]
    for loc_id, location in locations.items():
        if loc_id == "O":
            marker, label = "s", "Depot"
        elif loc_id.startswith("P"):
            marker, label = "^", "Pickup"
        elif loc_id.startswith("D"):
            marker, label = "v", "Drop-off"
        else:
            marker, label = "D", "Charging station"
        ax1.scatter(
            location.x,
            location.y,
            marker=marker,
            s=75,
            color="black",
            label=label,
            zorder=3,
        )
        ax1.annotate(
            loc_id,
            (location.x, location.y),
            xytext=(3, 3),
            textcoords="offset points",
            fontsize=8,
        )

    if best_solution is not None:
        colors = plt.cm.tab10(np.linspace(0, 1, max(len(agvs), 1)))
        for index, agv in enumerate(agvs):
            route = best_solution.routes[agv.id]
            if len(route) <= 2:
                continue
            for from_node, to_node in zip(route, route[1:]):
                from_location = locations[from_node]
                to_location = locations[to_node]
                linestyle = "--" if to_node.startswith("C") else "-"
                ax1.plot(
                    [from_location.x, to_location.x],
                    [from_location.y, to_location.y],
                    color=colors[index],
                    linestyle=linestyle,
                    linewidth=1.8,
                    alpha=0.8,
                    label=f"AGV {agv.id} route",
                )

    ax1.set_xlabel("X coordinate (distance units)")
    ax1.set_ylabel("Y coordinate (distance units)")
    ax1.set_title("Terminal locations and best feasible routes")
    ax1.grid(True, alpha=0.3)
    handles, labels = ax1.get_legend_handles_labels()
    unique_labels = dict(zip(labels, handles))
    ax1.legend(unique_labels.values(), unique_labels.keys(), loc="best", fontsize=8)

    # Cost uses the left axis; the feasible-ant count uses the right axis.
    ax2 = axes[0, 1]
    generations = [item["generation"] for item in convergence_history]
    best_costs = [
        item["best_cost"] if np.isfinite(item["best_cost"]) else np.nan
        for item in convergence_history
    ]
    average_costs = [item["avg_cost"] for item in convergence_history]
    feasible_counts = [item["feasible_count"] for item in convergence_history]
    line1 = ax2.plot(generations, best_costs, label="Best feasible cost")
    line2 = ax2.plot(generations, average_costs, "--", label="Average feasible cost")
    ax2_twin = ax2.twinx()
    line3 = ax2_twin.plot(
        generations,
        feasible_counts,
        color="tab:red",
        alpha=0.75,
        label="Feasible ants",
    )
    ax2.set_xlabel("Generation")
    ax2.set_ylabel("Travel-time cost")
    ax2_twin.set_ylabel("Feasible ants per generation")
    ax2.set_title("Feasibility and cost over generations")
    ax2.grid(True, alpha=0.3)
    lines = line1 + line2 + line3
    ax2.legend(lines, [line.get_label() for line in lines], loc="best")

    ax3 = axes[1, 0]
    ax3.plot(
        range(1, len(diversity_history) + 1),
        diversity_history,
        color="purple",
        linewidth=1.8,
    )
    ax3.set_xlabel("Generation")
    ax3.set_ylabel("Mean pairwise route-set distance")
    ax3.set_title("Solution diversity")
    ax3.grid(True, alpha=0.3)

    ax4 = axes[1, 1]
    methods = ["Simple reference", "ACO"]
    costs = [comparison_results["heuristic_cost"], comparison_results["aco_cost"]]
    violations = [
        comparison_results["heuristic_violations"],
        comparison_results["aco_violations"],
    ]
    positions = np.arange(len(methods))
    width = 0.35
    cost_bars = ax4.bar(
        positions - width / 2,
        costs,
        width,
        label="Travel-time cost",
        color="lightblue",
    )
    ax4_twin = ax4.twinx()
    violation_values = [
        value if value is not None else np.nan for value in violations
    ]
    violation_bars = ax4_twin.bar(
        positions + width / 2,
        violation_values,
        width,
        label="Battery-reserve violations",
        color="lightcoral",
    )
    ax4.set_xlabel("Plan source")
    ax4.set_ylabel("Travel-time cost")
    ax4_twin.set_ylabel("Battery-reserve violations")
    ax4.set_title("ACO and simple reference rule")
    ax4.set_xticks(positions)
    ax4.set_xticklabels(methods)
    ax4.legend([cost_bars, violation_bars], ["Travel-time cost", "Battery-reserve violations"])

    for bars, axis in [(cost_bars, ax4), (violation_bars, ax4_twin)]:
        for bar in bars:
            height = bar.get_height()
            if np.isfinite(height):
                axis.annotate(
                    f"{height:.1f}",
                    (bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha="center",
                    fontsize=8,
                )

    fig.tight_layout()
    plt.show()
    return fig


fig = visualize_aco_results(
    locations,
    agvs,
    best_solution,
    convergence_history,
    diversity_history,
    comparison_results,
)

## Pheromone-trail analysis

A large pheromone value indicates that an edge has received comparatively strong reinforcement during this run. It is not, by itself, proof that the edge belongs to a globally optimal plan: pheromone values reflect the history of this stochastic search, including its parameter settings and the feasible solutions it encountered. The heatmap below displays a selected subset of nodes; the printed ranking and summary statistics use the full pheromone table.

In [ ]:
def analyze_pheromone_trails(aco):
    """Summarize all pheromone trails and plot their distribution."""
    print("\n" + "=" * 60)
    print("PHEROMONE-TRAIL ANALYSIS")
    print("=" * 60)

    sorted_trails = sorted(
        aco.pheromone.items(), key=lambda item: item[1], reverse=True
    )
    print("Top 10 directed trails:")
    for rank, (edge, pheromone) in enumerate(sorted_trails[:10], start=1):
        print(f"  {rank:2d}. {edge[0]} -> {edge[1]}: {pheromone:.4f}")

    pheromone_values = np.array(list(aco.pheromone.values()))
    print("\nPheromone statistics:")
    print(f"  Mean: {np.mean(pheromone_values):.4f}")
    print(f"  Standard deviation: {np.std(pheromone_values):.4f}")
    print(f"  Minimum: {np.min(pheromone_values):.4f}")
    print(f"  Maximum: {np.max(pheromone_values):.4f}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    ax1.hist(
        pheromone_values,
        bins=20,
        color="skyblue",
        alpha=0.8,
        edgecolor="black",
    )
    ax1.set_xlabel("Pheromone level")
    ax1.set_ylabel("Number of directed edges")
    ax1.set_title("Pheromone-trail distribution")
    ax1.grid(True, alpha=0.3)

    key_nodes = [
        node for node in ["O", "P1", "P2", "P3", "D1", "D2", "D3", "C1", "C2"]
        if node in aco.nodes
    ]
    pheromone_matrix = np.zeros((len(key_nodes), len(key_nodes)))
    for row, node_i in enumerate(key_nodes):
        for column, node_j in enumerate(key_nodes):
            if node_i != node_j:
                pheromone_matrix[row, column] = aco.pheromone[(node_i, node_j)]

    image = ax2.imshow(pheromone_matrix, cmap="YlOrRd", aspect="auto")
    ax2.set_xticks(range(len(key_nodes)))
    ax2.set_xticklabels(key_nodes, rotation=45)
    ax2.set_yticks(range(len(key_nodes)))
    ax2.set_yticklabels(key_nodes)
    ax2.set_xlabel("Destination node")
    ax2.set_ylabel("Origin node")
    ax2.set_title("Directed pheromone levels for selected nodes")
    colorbar = plt.colorbar(image, ax=ax2)
    colorbar.set_label("Pheromone level")

    fig.tight_layout()
    plt.show()
    return sorted_trails[:10]


top_trails = analyze_pheromone_trails(aco)

## Summary and interpretation

### What the run measures

- **Best feasible cost:** the lowest total travel time among feasible plans found up to each generation. It is a best-so-far record and cannot increase during a run.
- **Average feasible cost:** the mean cost of feasible ant solutions in that generation. It is undefined when no feasible ant is found in that generation.
- **Feasible-ant count:** the number of constructed plans that respect every AGV's battery reserve on every travel leg.
- **Solution diversity:** the mean pairwise distance between per-AGV sets of visited non-depot nodes. Because route order is ignored, this metric captures assignment and node-set differences rather than every route difference.
- **Pheromone levels:** accumulated reinforcement on directed edges after evaporation and deposits from feasible plans.

### How to read the algorithm

Pheromone provides a shared search memory, while the heuristic favors locally attractive moves. The probability rule combines these influences, and the battery-aware factor steers construction toward charging options when a proposed travel leg threatens the reserve. The reserve remains a hard feasibility test in route evaluation. Only feasible solutions deposit pheromone, and the best feasible plan encountered is retained across generations.

### Performance and limitations

ACO is stochastic and does not guarantee a global optimum. A fixed random seed makes a fresh execution reproducible under the same software environment, but the output remains specific to the chosen instance, parameters, and search run. The simple reference rule is deliberately limited and is not a rigorous performance benchmark.

The implementation evaluates one route-construction process per AGV and ant, stores pheromone on directed node pairs, and computes pairwise diversity within each generation. Runtime and memory grow with the number of nodes, ants, and generations. The present example omits time-window feasibility, service and charging durations, station congestion, and vehicle interactions; conclusions should therefore be limited to the stated travel-time objective and battery-reserve model.

### Further study

Natural extensions within this modeling framework include incorporating the already-recorded time-window and service-time data into route evaluation, modeling finite charging duration, and testing larger terminal instances. Such extensions require updating the model assumptions, objective or feasibility checks, and interpretation of results together.